In [74]:
import os
import numpy as np
import polars as pl
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest

from snanomaly import dirs
import snanomaly.models.results.util as resutil
from snanomaly.models.results.util import explode_lists_to_numbered_columns
from snanomaly.models.sncandidate import Bandset
from snanomaly.preprocessing.band_transform import BandTransform

# Read data

In [5]:
dataset_name = "osc2018_june"

### Read non-reduced interpolated light curves

In [6]:
df_non_reduced = pl.read_parquet(dirs.INTERPOLATED / f"{dataset_name}_LSB-STATIC" / f"{dataset_name}.parquet")
df_non_reduced

sn_name,bandset,peak_time,days_pre_peak,days_post_peak,log_likelihood,thetas,pred_means,pred_stds
str,list[str],f64,i64,i64,f64,list[f64],list[list[f64]],list[list[f64]]
"""SDSS-II SN 17907""","[""g_pr"", ""r_pr"", ""i_pr""]",54360.5,20,100,40.859446,"[1.817474, 0.976793, … -0.112782]","[[1.0836e-9, 1.7216e-9, … 0.0], [1.5718e-9, 2.4974e-9, … 0.0], [1.5095e-9, 2.3984e-9, … 0.0]]","[[3.1032e-8, 3.1026e-8, … 0.0], [4.7504e-8, 4.7496e-8, … 0.0], [4.7784e-8, 4.7776e-8, … 0.0]]"
"""SDSS-II SN 15074""","[""g_pr"", ""r_pr"", ""i_pr""]",54019.5,20,100,20.305697,"[1.040616, 0.006335, … -0.185931]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0]]","[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0]]"
"""PS1-12bku""","[""g_pr"", ""r_pr"", ""i_pr""]",56206.5,20,100,5.712248,"[1.485435, 1.028905, … -0.212365]","[[3.1984e-9, 4.0253e-9, … 5.2086e-16], [4.3131e-9, 5.2261e-9, … 7.6584e-16], [5.2259e-9, 6.4279e-9, … 9.0266e-16]]","[[3.0114e-9, 2.6499e-9, … 3.5441e-9], [4.6124e-9, 4.0861e-9, … 5.3755e-9], [5.7363e-9, 5.1584e-9, … 6.5915e-9]]"
"""PS1-12bku""","[""g"", ""r"", ""i""]",56174.5,20,100,9.002704,"[0.378619, 0.112686, … -4.6510e-9]","[[1.1954e-14, 2.5198e-13, … 3.3133e-20], [1.8651e-14, 3.9316e-13, … 5.1691e-20], [2.8754e-14, 6.0613e-13, … 7.9693e-20]]","[[2.8726e-9, 2.8726e-9, … 2.8726e-9], [4.5551e-9, 4.5551e-9, … 4.5551e-9], [7.0226e-9, 7.0226e-9, … 7.0226e-9]]"
"""SDSS-II SN 20592""","[""g_pr"", ""r_pr"", ""i_pr""]",54405.5,20,100,26.328436,"[1.245873, 1.124316, … 0.192283]","[[1.3023e-8, 1.1623e-8, … 0.0], [2.2930e-8, 1.9697e-8, … 2.7915e-126], [2.4090e-8, 1.9549e-8, … 4.2657e-126]]","[[4.7809e-13, 1.8088e-10, … 0.0], [4.7809e-13, 1.7267e-9, … 1.4106e-8], [4.7809e-13, 3.8439e-9, … 2.1556e-8]]"
…,…,…,…,…,…,…,…,…
"""SN2016ayf""","[""g"", ""r"", ""i""]",57464.5,20,100,9.115077,"[0.00002, 0.000018, … 0.054203]","[[6.7301e-93, 1.9789e-84, … 0.0], [7.4106e-93, 2.1790e-84, … 0.0], [4.0208e-93, 1.1823e-84, … 0.0]]","[[0.000002, 0.000002, … 0.0], [0.000003, 0.000003, … 0.0], [0.000002, 0.000002, … 0.0]]"
"""SN2007ai""","[""g"", ""r"", ""i""]",54176.5,20,100,61.49233,"[1.616266, 1.287578, … 0.042188]","[[6.4079e-10, 1.1244e-9, … 7.3917e-12], [1.0794e-9, 1.8976e-9, … 1.2469e-11], [8.0077e-10, 1.4090e-9, … 9.2568e-12]]","[[2.1474e-8, 2.1465e-8, … 2.1478e-8], [3.7877e-8, 3.7862e-8, … 3.7884e-8], [2.9635e-8, 2.9624e-8, … 2.9640e-8]]"
"""PTF11dec""","[""g"", ""r"", ""i""]",55704.5,20,100,35.527299,"[2.277329, 1.5342, … 0.059556]","[[2.8156e-9, 3.3589e-9, … 0.0], [4.1483e-9, 4.9497e-9, … 0.0], [2.6185e-9, 3.1252e-9, … 0.0]]","[[7.3147e-9, 7.1975e-9, … 0.0], [1.0879e-8, 1.0708e-8, … 0.0], [7.2381e-9, 7.1353e-9, … 0.0]]"


### Read dimensionality-reduced data

In [71]:
dim_red_method = "TSNE"
dims = 2
df_reduced = pl.read_parquet(dirs.DIMREDUCED / dim_red_method / f"{dataset_name}_{dim_red_method}{dims}.parquet")
df_reduced

sn_name,bandset,values
str,list[str],list[f32]
"""SDSS-II SN 17907""","[""g_pr"", ""r_pr"", ""i_pr""]","[11.775729, -0.912904]"
"""SDSS-II SN 15074""","[""g_pr"", ""r_pr"", ""i_pr""]","[7.939235, -0.23015]"
"""PS1-12bku""","[""g_pr"", ""r_pr"", ""i_pr""]","[-45.086761, 13.524693]"
"""PS1-12bku""","[""g"", ""r"", ""i""]","[-13.744379, 7.033464]"
"""SDSS-II SN 20592""","[""g_pr"", ""r_pr"", ""i_pr""]","[-32.976837, 10.518064]"
…,…,…
"""SN2016ayf""","[""g"", ""r"", ""i""]","[-42.538879, -27.932192]"
"""SN2007ai""","[""g"", ""r"", ""i""]","[31.613644, -1.428256]"
"""PTF11dec""","[""g"", ""r"", ""i""]","[26.528658, -5.393352]"


# Isolation Forest

In [11]:
isoforest = IsolationForest(n_estimators=1000, max_samples=1572, contamination=0.02, random_state=42)

In [21]:
X = explode_lists_to_numbered_columns(df_reduced, "values").select(pl.exclude(["sn_name", "bandset"]))
X

values_0,values_1
f32,f32
11.775729,-0.912904
7.939235,-0.23015
-45.086761,13.524693
-13.744379,7.033464
-32.976837,10.518064
…,…
-42.538879,-27.932192
31.613644,-1.428256
26.528658,-5.393352


In [22]:
isoforest.fit(X)

IsolationForest(contamination=0.02, max_samples=1572, n_estimators=1000,
                random_state=42)

In [24]:
pred = isoforest.predict(X)
pred

array([1, 1, 1, ..., 1, 1, 1], shape=(1572,))

In [25]:
score = isoforest.score_samples(X)
score

array([-0.4655956 , -0.46100763, -0.53464558, ..., -0.46841271,
       -0.561278  , -0.47072059], shape=(1572,))

In [68]:
ind = sorted([i for i in range(pred.shape[0]) if pred[i] == -1], key=lambda x: score[x])
print("Outliers:")
out_df = df_reduced[ind].insert_column(3, pl.Series(name="anomaly_score", values=score[ind]))
out_df

Outliers:


sn_name,bandset,values,anomaly_score
str,list[str],list[f32],f64
"""SDSS-II SN 21386""","[""g_pr"", ""r_pr"", ""i_pr""]","[17.733162, 38.714638]",-0.628113
"""SDSS-II SN 6149""","[""g_pr"", ""r_pr"", ""i_pr""]","[-60.507946, -11.174562]",-0.626443
"""SDSS-II SN 14221""","[""g_pr"", ""r_pr"", ""i_pr""]","[-60.420364, -11.288226]",-0.621186
"""SDSS-II SN 2615""","[""g_pr"", ""r_pr"", ""i_pr""]","[54.833916, 8.463705]",-0.619367
"""SDSS-II SN 14552""","[""g_pr"", ""r_pr"", ""i_pr""]","[28.589369, 36.173809]",-0.615146
…,…,…,…
"""SN2009ib""","[""g_pr"", ""r_pr"", ""i_pr""]","[41.241882, -26.259424]",-0.584473
"""SDSS-II SN 13917""","[""g_pr"", ""r_pr"", ""i_pr""]","[6.288238, 34.011089]",-0.584407
"""SN2004dt""","[""g"", ""r"", ""i""]","[-5.415148, -29.303276]",-0.584373


In [75]:
outlier_method = "ISOFOREST"
out_dir = dirs.ANOMALIES / outlier_method
os.makedirs(out_dir, exist_ok=True)

In [77]:
out_path = out_dir / f"{dataset_name}_{dim_red_method}{dims}_{outlier_method}.parquet"
out_df.select(pl.exclude("values")).write_parquet(out_path)

In [78]:
pl.read_parquet(out_path)

sn_name,bandset,anomaly_score
str,list[str],f64
"""SDSS-II SN 21386""","[""g_pr"", ""r_pr"", ""i_pr""]",-0.628113
"""SDSS-II SN 6149""","[""g_pr"", ""r_pr"", ""i_pr""]",-0.626443
"""SDSS-II SN 14221""","[""g_pr"", ""r_pr"", ""i_pr""]",-0.621186
"""SDSS-II SN 2615""","[""g_pr"", ""r_pr"", ""i_pr""]",-0.619367
"""SDSS-II SN 14552""","[""g_pr"", ""r_pr"", ""i_pr""]",-0.615146
…,…,…
"""SN2009ib""","[""g_pr"", ""r_pr"", ""i_pr""]",-0.584473
"""SDSS-II SN 13917""","[""g_pr"", ""r_pr"", ""i_pr""]",-0.584407
"""SN2004dt""","[""g"", ""r"", ""i""]",-0.584373
